In [ ]:
import os
import pandas as pd
import cartopy.crs as ccrs
import cmcrameri  # To register colormaps for matplotlib e.g. cmc.batlow
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import geopandas as gpd
import pypsa
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import pandas as pd
import os
import matplotlib as mpl
from mpl_toolkits.axisartist.axislines import AxesZero
from mpl_toolkits.axisartist.parasite_axes import HostAxes, ParasiteAxes
import pandas as pd
import yaml

import extract_bus_information as extract_bus_information
import plot_hydrogen_network as plot_hydrogen_network

In [ ]:
def get_pgf_with_latex(scale_factor=1.0):
    base_axes_labelsize = 8.8
    base_font_size = 8.8
    base_axes_titlesize = 8.8
    base_legend_fontsize = 7.5
    base_xtick_labelsize = 7.5
    base_ytick_labelsize = 7.5

    return {
        "pgf.texsystem": "pdflatex",  # change this if using xetex or lautex
        "text.usetex": True,  # use LaTeX to write all text
        "font.family": "serif",
        # "svg.fonttype": 'times',
        "font.serif": ["times"],  # blank entries should cause plots
        "font.sans-serif": [],  # to inherit fonts from the document
        "font.monospace": [],
        "axes.labelsize": base_axes_labelsize
        * scale_factor,  # LaTeX default is 10pt font.
        "font.size": base_font_size * scale_factor,
        "axes.titlesize": base_axes_titlesize * scale_factor,
        "lines.linewidth": 2,
        "lines.markersize": 5,
        "legend.fontsize": base_legend_fontsize
        * scale_factor,  # Make the legend/label fonts
        # "legend.borderaxespad": 0.2,
        "xtick.labelsize": base_xtick_labelsize * scale_factor,  # a little smaller
        "ytick.labelsize": base_ytick_labelsize * scale_factor,
        "grid.linewidth": 0.5,
        "grid.alpha": 0.5,
        "pgf.preamble": "\n".join(
            [  # plots will use this preamble
                r"\usepackage[utf8]{inputenc}",
                r"\usepackage[T1]{fontenc}",
                r"\usepackage[detect-all,locale=DE]{siunitx}",
                r"\usepackage{eurosym}",
                r"\usepackage[version=4]{mhchem}",
                r"\sisetup{per-mode = symbol}",
                r"\DeclareSIUnit{\sieuro}{\mbox{\euro}}",
                r"\DeclareSIUnit{\a}{a}",
            ]
        ),
        "text.latex.preamble": "\n".join(
            [  # plots will use this preamble
                r"\usepackage[utf8]{inputenc}",
                r"\usepackage[T1]{fontenc}",
                r"\usepackage[detect-all,locale=DE]{siunitx}",
                r"\usepackage{eurosym}",
                r"\usepackage[version=4]{mhchem}",
                r"\sisetup{per-mode = symbol}",
                r"\DeclareSIUnit{\sieuro}{\mbox{\euro}}",
                r"\DeclareSIUnit{\a}{a}",
            ]
        ),
    }


photocatalysis_market_share_mapping_STH_10_UGHS = {
    "168.7": "0.0",
    "164.1": "4.3",
    "159.6": "5.1",
    "155.0": "9.6",
    "145.9": "20.3",
    "136.8": "31.1",
    "127.7": "42.8",
    "118.5": "51.2",
}


def get_base_index(df: pd.DataFrame):
    """
    Determines and separates the DataFrame index into different scenario
    categories, returning indices for the base scenario and scenario
    variations.

    Parameters
    ----------
    df : pd.DataFrame
        The summary DataFrame containing scenario indices.

    Returns
    -------
    tuple of pd.Index
        A tuple containing the indices for the base scenario, noUGHS variations,
        6pct variations, 3pct variations, and net_zero variations, respectively.
    """

    # Define scenario patterns and corresponding names
    scenario_patterns = {
        "noUGHS": "_noUGHS",
        "pct3": "_3pct",
        "pct6": "_6pct",
        "net_zero": "net_zero",
        "vopt": "_vopt",
        "wind_1_2": "_wind_1_2",
        "CCS_50": "_CCS_50",
        "CCS_100": "_CCS_100",
        "CCS_300": "_CCS_300",
        "CCS_400": "_CCS_400",
    }

    # Create a dictionary to hold the indices for each scenario
    scenario_indices = {
        key: df.index[df.index.str.contains(pattern)]
        for key, pattern in scenario_patterns.items()
    }

    # Calculate the base_index by removing all scenario indices
    base_index = df.index
    for indices in scenario_indices.values():
        base_index = base_index.difference(indices)

    # Return the indices in the desired order
    return (
        base_index,
        scenario_indices["noUGHS"],
        scenario_indices["pct6"],
        scenario_indices["pct3"],
        scenario_indices["net_zero"],
        scenario_indices["vopt"],
        scenario_indices["wind_1_2"],
        scenario_indices["CCS_50"],
        scenario_indices["CCS_100"],
        scenario_indices["CCS_300"],
        scenario_indices["CCS_400"],
    )


def set_size(use=True, fraction=1, subplots=(1, 1), scale=1):
    """
    Set figure dimensions to sit nicely in our document.

    Parameters
    ----------
    width_pt: float
            Document width in points
    fraction: float, optional
            Fraction of the width which you wish the figure to occupy
    subplots: array-like, optional
            The number of rows and columns of subplots.
    Returns
    -------
    fig_dim: tuple
            Dimensions of figure in inches
    """

    ## elsevier guide width
    # single column: 90 mm / 255 pt
    # 1.5 column: 140 mm / 397 pt
    # 2 column: 190 mm / 539 pt

    # Column width Nature Energy
    # (pt with values from above)
    # single column: 88 mm / 250 pt
    # 2 column: 180 mm / 511 pt

    # Width of figure (in pts)
    width_pt = 468
    height_pt = 622
    fig_width_pt = width_pt * fraction
    # Convert from pt to inches
    inches_per_pt = 1 / 72.27

    # Golden ratio to set aesthetic figure height
    golden_ratio = (5**0.5 - 1) / 2 * scale

    # Figure width in inches
    fig_width_in = fig_width_pt * inches_per_pt
    if use == True:
        # Figure height in inches
        fig_height_in = fig_width_in * golden_ratio * (subplots[0] / subplots[1])
    else:
        fig_height_in = fig_width_in * golden_ratio * 1.3 * (subplots[0] / subplots[1])

    return (fig_width_in, fig_height_in)


def save_or_plot_img(fig, output_folder_path, name, save=False):
    if save:
        fig.savefig(
            os.path.join(output_folder_path, f"{name}.svg"),
            format="svg",
            bbox_inches="tight",
        )
    else:
        fig.tight_layout()
        plt.show()

In [ ]:
#### ---- EL vs Mix


def plot_electrolysis_only_vs_balanced_mix_map(
    filename, info_dict, regions, tech_colors, output_folder_path, save=False
):
    n_left = pypsa.Network(
        os.path.join(
            "../results/raw/", info_dict["file_left"], "elec_s_150_lv1.25__I-H_2045.nc"
        )
    )
    try:
        n_right = pypsa.Network(
            os.path.join(
                "../results/raw/",
                info_dict["file_right"],
                "elec_s_150_lv1.25__I-H_2045.nc",
            )
        )
    except:
        n_right = pypsa.Network(
            os.path.join(
                "../results/raw/",
                info_dict["file_right"],
                "elec_s_150_lvopt__I-H_2045.nc",
            )
        )

    projection = ccrs.EqualEarth()

    nrows = 1
    ncols = 2
    fig, axes = plt.subplots(
        figsize=set_size(use=True, fraction=2, subplots=(nrows, ncols), scale=1.7),
        ncols=ncols,
        nrows=nrows,
        subplot_kw={"projection": projection},
    )

    network_map = {
        "el": {"n": n_left, "ax": axes[0], "title": info_dict["title_left"]},
        "mix": {"n": n_right, "ax": axes[1], "title": info_dict["title_right"]},
    }

    # cmap = "Blues"
    # cmap = "cmc.devon_r"
    cmap = "cmc.oslo_r"
    vmax = 3
    vmin = 0

    for idx, vals in enumerate(network_map.values()):
        (
            regions,
            bus_sizes,
            link_widths_total,
            link_widths_retro,
            bus_size_factor,
            linewidth_factor,
            new_network,
        ) = extract_bus_information.collect_bus_sizes(vals["n"], regions, projection)

        map_opts = {
            "boundaries": [-11, 30, 34, 71],
            "color_geomap": {"ocean": "white", "land": "whitesmoke"},
        }

        # Buses (Generators) and H2 pipelines
        vals["n"].plot(
            geomap=True,
            bus_sizes=bus_sizes,
            bus_colors=tech_colors,
            link_colors=tech_colors["H2 pipeline (total)"],
            link_widths=link_widths_total,
            branch_components=["Link"],
            ax=vals["ax"],
            **map_opts,
        )

        # Retrofitted pipelines (H2)
        vals["n"].plot(
            geomap=True,
            bus_sizes=0,
            link_colors=tech_colors["H2 pipeline (repurposed)"],
            link_widths=link_widths_retro,
            branch_components=["Link"],
            ax=vals["ax"],
            **map_opts,
        )

        # Hydrogen Storage regions
        regions.plot(
            ax=vals["ax"],
            column="H2",
            cmap=cmap,
            linewidths=0,
            vmax=vmax,
            vmin=vmin,
        )

        vals["ax"].set_facecolor("white")
        vals["ax"].set_title(vals["title"], fontsize=16, fontweight="bold")
        # Add gridlines with conditional label drawing
        gl = vals["ax"].gridlines(draw_labels=True)
        if idx == 0:
            gl.right_labels = True  # Disable right labels for the left plot
        elif idx == 1:
            gl.left_labels = False  # Disable left labels for the right plot

        # Change tick label color to soft gray
        gl.xlabel_style = {"color": "gray"}
        gl.ylabel_style = {"color": "gray"}

    # define a mappable based on which the colorbar will be drawn
    mappable = cm.ScalarMappable(norm=mcolors.Normalize(vmin, vmax), cmap=cmap)

    bbox = axes[1].get_position()
    a, b, c, d = bbox.bounds

    # define position and extent of colorbar
    #                     x_pos y_pos dx  dy
    cb_ax = fig.add_axes([0.95, b + b / 10, 0.014, d - b / 5])

    # draw colorbar
    cbar = fig.colorbar(mappable, cax=cb_ax, orientation="vertical")

    # Add a label to the colorbar
    cbar.set_label("Hydrogen storage in TWh", labelpad=10)

    # Bubble legend for generator size
    sizes = [50, 10]
    labels = [f"{s} GW" for s in sizes]
    sizes = [s / bus_size_factor * 1e3 for s in sizes]

    legend_kw = dict(
        loc="upper left",
        bbox_to_anchor=(0, -0.05),
        labelspacing=0.8,
        handletextpad=0,
        frameon=False,
    )

    plot_hydrogen_network.add_legend_circles(
        axes[0],
        sizes,
        labels,
        srid=n_left.srid,
        patch_kw=dict(facecolor="lightgrey"),
        legend_kw=legend_kw,
    )

    # Legend for pipeline size
    sizes = [30, 10]
    labels = [f"{s} GW" for s in sizes]
    scale = 1e3 / linewidth_factor
    sizes = [s * scale for s in sizes]

    legend_kw = dict(
        loc="upper left",
        bbox_to_anchor=(0.25, -0.05),
        frameon=False,
        labelspacing=0.8,
        handletextpad=1,
    )

    plot_hydrogen_network.add_legend_lines(
        axes[0],
        sizes,
        labels,
        patch_kw=dict(color="lightgrey"),
        legend_kw=legend_kw,
    )

    carriers = [
        carrier
        for carrier in tech_colors.keys()
        if carrier
        not in [
            "H2 Fuel Cell",
            "H2 turbine",
            "onwind",
            "offwind",
            "hydrogen",
            "electricity",
            "solar",
        ]
    ]
    colors = [tech_colors[c] for c in carriers]

    labels = carriers.copy()
    labels[1] = "H2 Photocatalysis"

    legend_kw = dict(
        loc="upper left",
        bbox_to_anchor=(0.54, -0.05),
        ncol=2,
        frameon=False,
    )

    plot_hydrogen_network.add_legend_patches(
        axes[0], colors, labels, legend_kw=legend_kw
    )

    fig.subplots_adjust(wspace=+0.1)

    save_or_plot_img(fig, output_folder_path, filename, save=save)

In [ ]:
pgf_with_latex = get_pgf_with_latex(scale_factor=1.0)
mpl.rcParams.update(pgf_with_latex)

el_only = "150_lv1.25_I_H_2045_3H_PC_925Euro"
mix = "150_lv1.25_I_H_2045_3H_PC_650Euro"
regions = gpd.read_file("../data/regions_onshore_elec_s_150.geojson").set_index("name")
with open(r"../src/colors.yaml") as stream:
    all_colors = yaml.safe_load(stream)
    tech_colors = all_colors["tech_colors"]
    noUGHS_line_color = all_colors["others"]["no_UGHS_line"]
    UGHS_line_color = all_colors["others"]["UGHS_line"]
    colors_dict = all_colors["others"]

output_folder_path = r"../img/test/"


vs_plot_map = {
    "pc-0-wind_3_vs_wind_1_2": {
        "title_left": "PC-0 case; Wind 3.0 MW/km² (base)",
        "file_left": el_only,
        "title_right": "PC-0 case; Wind 1.2 MW/km²",
        "file_right": "150_lv1.25_I_H_2045_3H_PC_925Euro_wind_1_2",
    },
    "pc-50-wind_3_vs_wind_1_2": {
        "title_left": "PC-50 case; Wind 3.0 MW/km² (base)",
        "file_left": mix,
        "title_right": "PC-50 case; Wind 1.2 MW/km²",
        "file_right": "150_lv1.25_I_H_2045_3H_PC_650Euro_wind_1_2",
    },
}

for filename in vs_plot_map.keys():
    plot_electrolysis_only_vs_balanced_mix_map(
        filename,
        vs_plot_map[filename],
        regions,
        tech_colors,
        output_folder_path,
        save=False,
    )

In [ ]:
mapping = {
    "PC-0": r"results\raw\150_lv1.25_I_H_2045_3H_PC_925Euro\elec_s_150_lv1.25__I-H_2045.nc",
    "PC-50": r"results\raw\150_lv1.25_I_H_2045_3H_PC_650Euro\elec_s_150_lv1.25__I-H_2045.nc",
    "'PC-0' wind 1.2": r"results\raw\150_lv1.25_I_H_2045_3H_PC_925Euro_wind_1_2\elec_s_150_lv1.25__I-H_2045.nc",
    "'PC-50' wind 1.2": r"results\raw\150_lv1.25_I_H_2045_3H_PC_650Euro_wind_1_2\elec_s_150_lv1.25__I-H_2045.nc",
    "Hofmann 2025 baseline": r"results\hofmann_2025_baseline.nc",
}

carriers = ["onwind", "offwind-dc", "offwind-ac", "solar", "photocatalysis"]

# Initialize a dictionary to hold the results
summary_dict = {label: {} for label in mapping.keys()}

for label, file in mapping.items():
    n = pypsa.Network(os.path.join("../", file))
    summary_dict[label]["node count"] = n.buses[n.buses.carrier == "AC"].shape[0]
    for carrier in carriers:
        try:
            filtered_df = n.generators[n.generators.carrier == carrier][
                ["p_nom_opt", "p_nom_max"]
            ]
            close_mask = np.isclose(filtered_df["p_nom_opt"], filtered_df["p_nom_max"])
            matches = filtered_df[close_mask]
            summary_dict[label][carrier] = len(matches)
        except:
            summary_dict[label][carrier] = "-"

summary_df = pd.DataFrame.from_dict(summary_dict, orient="index")


print(summary_df)
# summary_df.to_csv("../results/capacity_limit_reaches.csv")

# CO2 sequestration limits

In [ ]:
def plot_system_cost_differences_bars(
    n_1_name,
    n_2_name,
    output_folder_path,
    scale,
    y_heating,
    y_synfuels_lower,
    y_synfuels_upper,
    y_h2_lower,
    y_h2_upper,
    suffix,
    legend_labels,
    save=False,
):
    n_base = pypsa.Network(
        os.path.join("../results/raw/", n_1_name, "elec_s_150_lv1.25__I-H_2045.nc")
    )
    n_ccs_changed = pypsa.Network(
        os.path.join("../results/raw/", n_2_name, "elec_s_150_lv1.25__I-H_2045.nc")
    )

    from system_cost_differences import (
        calc_energy_box_values,
        calc_hydrogen_box,
        calc_power_and_h2_system_box_values,
        calc_syn_fuel_box_values,
    )

    # Calculate all required data
    (
        df_energy_carriers,
        x_energy_carriers,
        name_map_energy_carriers,
    ) = calc_energy_box_values(n_base, n_ccs_changed)
    df_syn_fuels, x_syn_fuels, name_map_syn_fuels = calc_syn_fuel_box_values(
        n_base, n_ccs_changed
    )
    df_hydrogen_system, x_hydrogen_system, hydrogen_system_name_map = calc_hydrogen_box(
        n_base, n_ccs_changed
    )
    (
        df_power_and_heating_system,
        x_power_and_heating_system,
        power_and_heating_system_name_map,
    ) = calc_power_and_h2_system_box_values(n_base, n_ccs_changed)

    with open(r"../src/colors.yaml") as stream:
        colors_dict = yaml.safe_load(stream)["others"]

    rotation_deg = 0
    width = 0.35  # Width of the bars

    # Define the layout using a list of strings

    # fmt: off
    layout = [
        ["A","A","A","A","A","A","A","A","A","A",],
        ["C","C","C","C","C","C","C","C","C","C",],
        ["B","B","B",".",".","D","D","D",".","E",],
    ]
    # fmt: on

    def create_bar_plot(ax, x_values, df, name_map, title, y_label_left, ax1_ylim):
        ax.set_title(title)

        # Plot the Installation bars on ax
        ax.bar(
            x_values - width / 2,
            df["Installation"],
            width,
            label="Installation",
            color=colors_dict["generation installation"],
        )
        ax.set_ylabel(y_label_left, color=colors_dict["generation installation"])
        ax.tick_params(axis="y", labelcolor=colors_dict["generation installation"])

        # Create a secondary y-axis for the stacked bars
        ax2 = ax.twinx()
        # Stack Invest and Opex on top of each other
        ax2.bar(
            x_values + width / 2,
            df["Invest"],
            width,
            label="Invest",
            color=colors_dict["annualized costs"],
        )
        ax2.bar(
            x_values + width / 2,
            df["Opex"],
            width,
            label="Opex",
            color=colors_dict["annualized costs"],
            bottom=df["Invest"],
        )

        # Add labels and titles
        ax2.set_ylabel(
            "Difference in annualized\ncosts in Bln. € a$^{-1}$",
            color=colors_dict["annualized costs"],
        )
        ax2.tick_params(axis="y", labelcolor=colors_dict["annualized costs"])
        ax.set_xticks(x_values)
        ax.set_xticklabels(
            [name_map[key] for key in df.index],
            rotation=rotation_deg,
        )

        # Fix axis spread
        ax.set_ylim(ax1_ylim)
        ax.set_xlim([-0.5, len(df) - 0.5])
        ax2.set_ylim(ax1_ylim / 15)

        # Grid and tick customization
        ax.grid(alpha=0.3)
        ax.tick_params(axis="both", direction="in", bottom=True, top=True, left=True)
        ax2.tick_params(axis="both", direction="in", right=True)
        ax.get_yaxis().set_major_formatter(
            mpl.ticker.FuncFormatter(lambda x, p: format(int(x), ","))
        )

        return ax2

    def setup_axes_zero(fig, position):
        """
        Creates and configures an AxesZero on the given figure at the specified
        position.
        """
        ax = fig.add_axes(position, axes_class=AxesZero)
        for direction in ["xzero", "yzero"]:
            ax.axis[direction].set_axisline_style("-|>")
            ax.axis[direction].set_visible(True)

        for direction in ["left", "right", "bottom", "top"]:
            ax.axis[direction].set_visible(False)

        ax.set_ylim([-2.2, 5.2])
        ax.set_yticks([])
        ax.set_xticks([])
        return ax

    def plot_bars(ax):
        """
        Plots bar data on the given axis.
        """

        ax.bar(
            x=[0.2, 1.2, 2.2, 3.8, 4.8, 5.8],
            height=[5, 3, 2, 5, 7, 2],
            width=[0.9] * 6,
            bottom=[0, 2, 0, 0, -2, -2],
            align="edge",
            color=[
                colors_dict["pc-50"],
                colors_dict["pc-0"],
                colors_dict["difference"],
            ],
            label=["PC-50", "PC-0", "Difference", "", "", ""],
        )

    def plot_lines_and_arrows(ax):
        """
        Adds lines and arrows annotations to the given axis.
        """
        lw = 1.5
        lines = [
            ([0.2, 2.1], [5, 5]),
            ([1.2, 3.1], [2, 2]),
            ([3.8, 5.7], [5, 5]),
            ([4.8, 6.7], [-2, -2]),
        ]
        for x, y in lines:
            ax.plot(x, y, "k", linewidth=lw)

        arrows = {
            "bm-1": {"xy": (0.65, 0), "xyt": (0.65, 5), "ls": "--", "a": 0.5},
            "el-1": {"xy": (1.65, 5), "xyt": (1.65, 2), "ls": "--", "a": 0.5},
            "d-1": {"xy": (2.65, 0), "xyt": (2.65, 2), "ls": "-", "a": 1},
            "bm-2": {"xy": (4.25, 0), "xyt": (4.25, 5), "ls": "--", "a": 0.5},
            "el-2": {"xy": (5.25, 5), "xyt": (5.25, -2), "ls": "--", "a": 0.5},
            "d-2": {"xy": (6.25, 0), "xyt": (6.25, -2), "ls": "-", "a": 1},
        }
        for v in arrows.values():
            ax.annotate(
                "",
                xy=v["xy"],
                xytext=v["xyt"],
                color="k",
                arrowprops=dict(
                    arrowstyle="<-",
                    linestyle=v["ls"],
                    alpha=v["a"],
                    color="k",
                ),
            )

    # Create the mosaic plot
    nrows = 3
    ncols = 7
    fig, ax_dict = plt.subplot_mosaic(
        layout,
        figsize=set_size(use=True, subplots=(nrows, ncols), fraction=1, scale=scale),
    )

    create_bar_plot(
        ax=ax_dict["A"],
        x_values=x_energy_carriers,
        df=df_energy_carriers,
        name_map=name_map_energy_carriers,
        title="Electricity and hydrogen production",
        y_label_left="Difference in energy\ngeneration in TWh a$^{-1}$",
        ax1_ylim=np.array([-1000, 1300]),
    )

    create_bar_plot(
        ax=ax_dict["B"],
        x_values=x_hydrogen_system,
        df=df_hydrogen_system,
        name_map=hydrogen_system_name_map,
        title="Hydrogen system",
        y_label_left="Difference in storage capacity in TWh\nand transport capacity in TWkm ",
        ax1_ylim=np.array([y_h2_lower, y_h2_upper]),
    )

    create_bar_plot(
        ax=ax_dict["C"],
        x_values=x_power_and_heating_system,
        df=df_power_and_heating_system,
        name_map=power_and_heating_system_name_map,
        title="Heating and power system",
        y_label_left="Difference in installed capacity\nin GW and TWh (storage)",
        ax1_ylim=np.array([-y_heating, y_heating]),
    )

    ax2 = create_bar_plot(
        ax=ax_dict["D"],
        x_values=x_syn_fuels,
        df=df_syn_fuels,
        name_map=name_map_syn_fuels,
        title="Synthetic fuels",
        y_label_left="Difference in generation in TWh a$^{-1}$\nand Mt CO$_2$ a$^{-1}$ (DAC)",
        ax1_ylim=np.array([y_synfuels_lower, y_synfuels_upper]),
    )

    # Show the plot
    handles1, labels1 = ax_dict["A"].get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    labels = ["Yearly generation / total installation", "Annualized costs per year"]
    fig.legend(
        handles1 + [handles2[0]],
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=3,
        frameon=False,
        fancybox=False,
    )

    plt.subplots_adjust(left=0, bottom=0.07, right=1, top=0.9, wspace=0.0, hspace=0.28)

    ### Add explanatory legend plot
    # Define bounding position and remove the placeholder axis
    bounds_orig = ax_dict["E"].get_position(fig).bounds
    bounds = (0.925, 0.085, 0.15, 0.18)
    fig.delaxes(ax_dict["E"])

    # # Create and set up AxesZero
    ax_explain = setup_axes_zero(fig, bounds)
    # # Plot components
    plot_bars(ax_explain)
    plot_lines_and_arrows(ax_explain)
    ax_explain.set_title("Difference\nexplanation")

    # # Add legend
    ax_explain.legend(
        [legend_labels[0], legend_labels[-1], "Difference"],
        loc="upper center",
        bbox_to_anchor=(0.5, 0),
        frameon=False,
        fancybox=False,
    )

    # Draw a rectangle on the entire figure (not inside any specific axis)
    rect = plt.Rectangle(
        # (bounds[0] - 0.015, bounds[1] - 0.081),
        (bounds[0] - 0.015, bounds[1] - 0.071),
        bounds[2] + 0.04,
        bounds[-1] + 0.124,
        transform=fig.transFigure,
        facecolor="white",
        edgecolor="black",
        fill=False,
    )
    fig.patches.append(rect)

    save_or_plot_img(
        fig, output_folder_path, f"system_cost_difference_{suffix}", save=save
    )

In [ ]:
filepath = os.path.join(r"../results/total_summary.csv")

# Load dataframe but skip the unit column
total_summary_df = pd.read_csv(filepath, index_col=0, header=0, skiprows=range(1, 2))


plot_mapping = {
    "PC0_200Mt_vs_50Mt": {
        "file1": "150_lv1.25_I_H_2045_3H_PC_925Euro",
        "file2": "150_lv1.25_I_H_2045_3H_PC_925Euro_CCS_50",
        "y_heating": 200,
        "y_synfuels_lower": -60,
        "y_synfuels_upper": 220,
        "y_h2_lower": -5,
        "y_h2_upper": 50,
        "legend_labels": ["'PC-0', CCS 50", "PC-0 (CCS 200)"],
    },
    "PC50_200Mt_vs_50Mt": {
        "file1": "150_lv1.25_I_H_2045_3H_PC_650Euro",
        "file2": "150_lv1.25_I_H_2045_3H_PC_650Euro_CCS_50",
        "y_heating": 300,
        "y_synfuels_lower": -60,
        "y_synfuels_upper": 220,
        "y_h2_lower": -5,
        "y_h2_upper": 50,
        "legend_labels": ["'PC-50', CCS 50", "PC-50 (CCS 200)"],
    },
    "PC50_200Mt_vs_400Mt": {
        "file1": "150_lv1.25_I_H_2045_3H_PC_650Euro",
        "file2": "150_lv1.25_I_H_2045_3H_PC_650Euro_CCS_400",
        "y_heating": 300,
        "y_synfuels_lower": -700,
        "y_synfuels_upper": 100,
        "y_h2_lower": -50,
        "y_h2_upper": 5,
        "legend_labels": ["'PC-50', CCS 400", "PC-50 (CCS 200)"],
    },
}


output_folder_path = r"../img/test/"

# Manually slightly changed in inkscape
for suffix, info_dict in plot_mapping.items():
    plot_system_cost_differences_bars(
        info_dict["file1"],
        info_dict["file2"],
        output_folder_path,
        scale=5,
        y_heating=info_dict["y_heating"],
        y_synfuels_lower=info_dict["y_synfuels_lower"],
        y_synfuels_upper=info_dict["y_synfuels_upper"],
        y_h2_lower=info_dict["y_h2_lower"],
        y_h2_upper=info_dict["y_h2_upper"],
        legend_labels=info_dict["legend_labels"],
        suffix=suffix,
        save=False,
    )

In [ ]:
pgf_with_latex = get_pgf_with_latex(scale_factor=1.5)
mpl.rcParams.update(pgf_with_latex)

total_summary_df = pd.read_csv(
    "../results/total_summary.csv",
    index_col=0,
    header=0,
    skiprows=range(1, 2),
)
(
    base_index,
    noUGHS_index,
    pct6_index,
    pct3_index,
    net_zero_index,
    vopt_index,
    wind_1_2_index,
    CCS_50_index,
    CCS_100_index,
    CCS_300_index,
    CCS_400_index,
) = get_base_index(total_summary_df)


fig, ax = plt.subplots(figsize=(3, 6))

ax.scatter(
    [
        total_summary_df.loc[base_index[0], "pc total annual cost"],
        total_summary_df.loc[base_index[-1], "pc total annual cost"],
    ],
    [
        total_summary_df.loc[CCS_400_index[0], "pc market share"],
        total_summary_df.loc[CCS_400_index[-1], "pc market share"],
    ],
    c="#733957",
    marker="+",
    s=120,
    label="400",
)
ax.scatter(
    [
        total_summary_df.loc[base_index[0], "pc total annual cost"],
        total_summary_df.loc[base_index[-1], "pc total annual cost"],
    ],
    [
        total_summary_df.loc[CCS_300_index[0], "pc market share"],
        total_summary_df.loc[CCS_300_index[-1], "pc market share"],
    ],
    c="#A3672C",
    marker="x",
    s=120,
    label="300",
)
ax.scatter(
    [
        total_summary_df.loc[base_index[0], "pc total annual cost"],
        total_summary_df.loc[base_index[-1], "pc total annual cost"],
    ],
    [
        total_summary_df.loc[base_index[0], "pc market share"],
        total_summary_df.loc[base_index[-1], "pc market share"],
    ],
    c="#D6D893",
    marker="o",
    s=30,
    label="200 (base)",
)
ax.scatter(
    [
        total_summary_df.loc[base_index[0], "pc total annual cost"],
        total_summary_df.loc[base_index[-1], "pc total annual cost"],
    ],
    [
        total_summary_df.loc[CCS_100_index[0], "pc market share"],
        total_summary_df.loc[CCS_100_index[-1], "pc market share"],
    ],
    c="#74BBCD",
    marker="x",
    s=120,
    label="100",
)
ax.scatter(
    [
        total_summary_df.loc[base_index[0], "pc total annual cost"],
        total_summary_df.loc[base_index[-1], "pc total annual cost"],
    ],
    [
        total_summary_df.loc[CCS_50_index[0], "pc market share"],
        total_summary_df.loc[CCS_50_index[-1], "pc market share"],
    ],
    c="#5C538B",
    marker="x",
    s=120,
    label="50",
)


el_100_x = total_summary_df.loc[
    f"150_lv1.25_I_H_2045_3H_PC_925Euro", "pc total annual cost"
]
el_100_y = total_summary_df.loc[f"150_lv1.25_I_H_2045_3H_PC_925Euro", "pc market share"]
balanced_mix_x = total_summary_df.loc[
    f"150_lv1.25_I_H_2045_3H_PC_650Euro", "pc total annual cost"
]
balanced_mix_y = total_summary_df.loc[
    f"150_lv1.25_I_H_2045_3H_PC_650Euro", "pc market share"
]
ax.annotate(
    "PC-0",
    xy=(el_100_x, el_100_y),
    xytext=(el_100_x - 15, el_100_y + 1.5),
    arrowprops=dict(arrowstyle="->", lw=1.5),
    ha="center",
)
ax.annotate(
    "PC-50",
    xy=(balanced_mix_x, balanced_mix_y),
    xytext=(135, 49),
    arrowprops=dict(arrowstyle="->", lw=1.5),
    ha="left",
)

ax.set_xlabel("Annualized photocatalysis cost\nin € a$^{-1}$ kW$^{-1}$")
ax.set_ylabel(r"Photocatalysis market share in \%")
ax.legend(title="CO$_2$ sequestration\npotential in Mt/a")
ax.invert_xaxis()
ax.grid()

# fig.savefig(
#     r"../img/CCS_scatter.svg",
#     format="svg",
#     bbox_inches="tight",
# )

In [ ]:
from plots import prepare_LCOH_violin_data

base_index, *_ = get_base_index(total_summary_df)

tot_caps = prepare_LCOH_violin_data(
    df=total_summary_df,
    index=base_index,
)

photocatalysis_market_share_mapping_STH_10_UGHS = {
    "168.7": "0.0",
    "164.1": "4.3",
    "159.6": "5.1",
    "155.0": "9.6",
    "145.9": "20.3",
    "136.8": "31.1",
    "127.7": "42.8",
    "118.5": "51.2",
}

median_df_el = (
    tot_caps[tot_caps["Technology"] == "Electrolysis"]
    .groupby("Annualized photocatalysis cost in € a$^{-1}$ kW$^{-1}$")[
        "$LCOH$ in € kg$^{-1}$"
    ]
    .median()
    .reset_index()
)
median_df_pc = (
    tot_caps[tot_caps["Technology"] == "Photocatalysis"]
    .groupby("Annualized photocatalysis cost in € a$^{-1}$ kW$^{-1}$")[
        "$LCOH$ in € kg$^{-1}$"
    ]
    .median()
    .reset_index()
)
overall_medians = median_df_el.copy()
overall_medians.rename(
    columns={"$LCOH$ in € kg$^{-1}$": "Electrolysis $LCOH$ in € kg$^{-1}$"},
    inplace=True,
)
overall_medians["Photocatalysis $LCOH$ in € kg$^{-1}$"] = median_df_pc[
    "$LCOH$ in € kg$^{-1}$"
]

mapping_float_keys = {
    float(k): float(v)
    for k, v in photocatalysis_market_share_mapping_STH_10_UGHS.items()
}

overall_medians["Market share"] = overall_medians[
    "Annualized photocatalysis cost in € a$^{-1}$ kW$^{-1}$"
].map(mapping_float_keys)
overall_medians = overall_medians[
    [
        "Market share",
        "Annualized photocatalysis cost in € a$^{-1}$ kW$^{-1}$",
        "Electrolysis $LCOH$ in € kg$^{-1}$",
        "Photocatalysis $LCOH$ in € kg$^{-1}$",
    ]
]
overall_medians.sort_values(by="Market share", inplace=True)
overall_medians["Percentual lower Photocatalysis $LCOH$"] = (
    (
        overall_medians["Electrolysis $LCOH$ in € kg$^{-1}$"]
        - overall_medians["Photocatalysis $LCOH$ in € kg$^{-1}$"]
    )
    / overall_medians["Electrolysis $LCOH$ in € kg$^{-1}$"]
    * 100
)
overall_medians.reset_index(inplace=True, drop=True)
overall_medians